<a href="https://colab.research.google.com/github/079bce159sarwagya-ctrl/Pytorch/blob/main/1.%20Foundation/Neural_Networks(NN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Autograd:

A core component of Pytorch that provides differentiation for tensor operations.

In [9]:
from math import e
import torch
import time
start=time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"
x=torch.tensor(3.0, requires_grad=True).to(device) #Tells pytorch to track operations on x
y=x**2

print(f"X: {x}")
print(f"Y: {y}")
y.backward()
print(f"X.grad from y: {x.grad}")
x.grad.zero_() # To inplace by zero in x.grad to remove previoulsy store gradient value of y in it
y=x**2
z=y+2
z.backward()
print(f"X.grad from z: {x.grad}")
end=time.time()
print(f"Time taken: {end-start}")

X: 3.0
Y: 9.0
X.grad from y: 6.0
X.grad from z: 6.0
Time taken: 0.005283832550048828


LOSS CALCULATION


In [9]:
#Manual loss calculation:

import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")


Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [8]:
#Loss calculation using Autograd

x = torch.tensor(6.7)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
z = w*x + b

y_pred = torch.sigmoid(z)
y_pred
loss = binary_cross_entropy_loss(y_pred, y)
loss
loss.backward()
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Clearing Grad

In [11]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x**2).mean() # y= f(x1,x2,x3)
print(y)
y.backward()
print(x.grad)
# Grad zero
x = torch.tensor(2.0, requires_grad=True)
print(x)
y = x ** 2
print(y)
y.backward()
print(x.grad)
x.grad.zero_()
print(x.grad)

tensor(4.6667, grad_fn=<MeanBackward0>)
tensor([0.6667, 1.3333, 2.0000])
tensor(2., requires_grad=True)
tensor(4., grad_fn=<PowBackward0>)
tensor(4.)
tensor(0.)


In [ ]:
# disable (gradient tracking / Backward tracking)
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward()
print(x.grad)

# option 1 - requires_grad_(False)
x.requires_grad_(False)
print(x.requires_grad)
# option 2 - detach()
z=x.detach() # z has same value as x but tracking is disabled
# option 3 - torch.no_grad()
with torch.no_grad():
    y = x ** 2
    print(y)
